In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

In [3]:
from sklearn.ensemble import RandomForestRegressor
from src import calculate_metrics

In [4]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("Random_Forest_Regressor_HPO")
mlflow.set_tracking_uri("http://localhost:5000")

In [6]:
import json
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, HalvingRandomSearchCV

In [6]:
param_list = {}

## Preparacion de datos

In [9]:
df = pd.read_csv('../data/train/transformed_train.csv')

y = df["Precio"]
X = df.drop(columns=["idx", "Título", "Descripción", "Precio"])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means) 

one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])

imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11682, 61)
Val: (2921, 61)


## First Experiment

Ajuste de hiperparametros

In [11]:
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'], 
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=42)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 50 candidates, totalling 150 fits


/home/elianaostro/miniconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.84e+13   |   5.69e+13
R²           |   0.9630     |   0.8834
MAE          |   1.24e+06   |   2.94e+06
EVS          |   0.9630     |   0.8834
Max Error    |   2.64e+08   |   2.44e+08
Median AE    |   6.17e+05   |   1.54e+06

Best parameters found:
{
    "n_estimators": 300,
    "min_samples_split": 10,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "max_depth": 40,
    "bootstrap": false
}


In [ ]:

param_dist = {
    'n_estimators': [200, 300, 400],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True],
    'max_samples': [0.6, 0.8, 1.0],  
    'criterion': ['squared_error', 'absolute_error'],
    'ccp_alpha': [0.0, 0.001, 0.01]
}


rf = RandomForestRegressor(random_state=42)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["RandomizedSearchCV_2"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.66e+13   |   5.89e+13
R²           |   0.9665     |   0.8794
MAE          |   1.45e+06   |   3.06e+06
EVS          |   0.9665     |   0.8794
Max Error    |   2.62e+08   |   2.45e+08
Median AE    |   8.10e+05   |   1.61e+06

Best parameters found:
{
    "n_estimators": 300,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_samples": 1.0,
    "max_features": "sqrt",
    "max_depth": 20,
    "criterion": "absolute_error",
    "ccp_alpha": 0.01,
    "bootstrap": true
}


In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv  
from sklearn.model_selection import HalvingRandomSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 5, 7, 9, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
    'bootstrap': [True],
    'criterion': ['squared_error', 'absolute_error'] 
}

rf = RandomForestRegressor(random_state=42)

halving_search = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    factor=3,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

halving_search.fit(X_train, y_train)

best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["HalvingRandomSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


n_iterations: 7
n_required_iterations: 7
n_possible_iterations: 7
min_resources_: 6
max_resources_: 11682
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 1080
n_resources: 6
Fitting 3 folds for each of 1080 candidates, totalling 3240 fits


/home/elianaostro/miniconda3/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1080 is smaller than n_iter=1947. Running 1080 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


----------
iter: 1
n_candidates: 360
n_resources: 18
Fitting 3 folds for each of 360 candidates, totalling 1080 fits
----------
iter: 2
n_candidates: 120
n_resources: 54
Fitting 3 folds for each of 120 candidates, totalling 360 fits
----------
iter: 3
n_candidates: 40
n_resources: 162
Fitting 3 folds for each of 40 candidates, totalling 120 fits
----------
iter: 4
n_candidates: 14
n_resources: 486
Fitting 3 folds for each of 14 candidates, totalling 42 fits
----------
iter: 5
n_candidates: 5
n_resources: 1458
Fitting 3 folds for each of 5 candidates, totalling 15 fits
----------
iter: 6
n_candidates: 2
n_resources: 4374
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   2.62e+13   |   5.35e+13
R²           |   0.9472     |   0.8904
MAE          |   1.45e+06   |   2.75e+06
EVS          |   0.9472     |   0.8904
Max Error    |   3.41e+08   |   2.37e+08
Median AE    |   6.78e+05

In [16]:
input_example = X_train.iloc[0:1]

for name, params in param_list.items():
    model = RandomForestRegressor(random_state=42, **params)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    metrics = calculate_metrics(y_val, y_pred_val, verbose=False)

    run_name = f"RandomForest_{name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        signature = infer_signature(input_example, model.predict(input_example))
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example, signature=signature)

2025/06/30 22:43:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForest_RandomizedSearchCV at: http://localhost:5000/#/experiments/995359195463226246/runs/5cb552cb1f3148fa8b6907c3c75983ec
🧪 View experiment at: http://localhost:5000/#/experiments/995359195463226246


2025/06/30 22:49:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForest_RandomizedSearchCV_2 at: http://localhost:5000/#/experiments/995359195463226246/runs/7c220ec4d8fa44259483be157316f89d
🧪 View experiment at: http://localhost:5000/#/experiments/995359195463226246


2025/06/30 23:01:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForest_HalvingRandomSearchCV at: http://localhost:5000/#/experiments/995359195463226246/runs/452b6e2c81f54eb4bb64941ce09d50a5
🧪 View experiment at: http://localhost:5000/#/experiments/995359195463226246


## Modelo despues de data transformamtion

In [ ]:
df_train = pd.read_csv('../data/train/transformed_train.csv')
df_val = pd.read_csv('../data/train/transformed_val.csv')

X_train = df_train.drop(columns=["Precio"])
y_train = df_train["Precio"]

X_val = df_val.drop(columns=["Precio"])
y_val = df_val["Precio"]


param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'], 
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=42)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

In [ ]:
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   9.98e+12   |   3.79e+13
R²           |   0.9833     |   0.9293
MAE          |   7.96e+05   |   2.82e+06
EVS          |   0.9833     |   0.9293
Max Error    |   2.20e+08   |   1.50e+08
Median AE    |   3.71e+05   |   1.46e+06

Best parameters found:
{
    "n_estimators": 400,
    "min_samples_split": 5,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "max_depth": null,
    "bootstrap": false
}
